#  Week 10, Day 1 Lab — Build Your First AI Agent
### From a stateless chatbot to a working ReAct loop, in 3 hours

**This is a build lab, not a run-and-watch lab.** After Section 2's one worked example, you will
write every tool, every schema, and the agent loop itself from a spec — not fill in one blank in
code that's already written for you.

**How to use this notebook:**
- Cells marked **✍️ WRITE THIS** have only a function signature / docstring / comments — you write
  the body.
- Cells marked **🔧 TODO** ask you to modify or extend something that's partially there.
- Plain cells (no marker) are given to you — API setup, test calls, etc. — run them as-is.


---
## 🛠️ Section 0 — Setup

We'll use **Gemini 2.5 Flash** via Google's `google-genai` SDK — free, no credit card required.

**Get your free API key (2 minutes):**
1. Go to **https://aistudio.google.com/apikey**
2. Sign in with any Google account
3. Click **Create API key** → copy it

> **Free tier note:** there's a rate limit (a handful of requests/minute). If a cell errors with
> a rate-limit message, wait ~30-60 seconds and re-run it.


In [1]:
# Install Google's official GenAI SDK
!pip install -q google-genai


In [2]:
import os
import json
import getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")


Enter your Gemini API key: ··········


In [5]:
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-3.6-flash" # I got an error using the model "gemini-2.6-flash" the error was ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.6-flash for the latest features and improvements. We recommend you to use the Interactions API.', 'status': 'NOT_FOUND'}} that is why I changed my model name.

# Sanity check — if this prints a reply, you're good to go
test = client.models.generate_content(
    model=MODEL,
    contents="Say 'Agent lab is ready!' and nothing else."
)
print(test.text)


Agent lab is ready!


---
##  Section 1 — Why Do We Need Agents? (15 minutes)

Run the cell below and see what happens when you ask a plain LLM call something it can't know.


In [6]:
response = client.models.generate_content(
    model=MODEL,
    contents="What's the exact current temperature in Lahore right now?"
)
print(response.text)


I do not have access to real-time information, so I cannot provide the current temperature in Lahore right now. Please check a live weather service or app for the latest update.


Did the model admit it doesn't know, guess, or
make something up? What would it need to answer correctly?




In [7]:
# No the model didn't guess anything that it doesn't know and it doesn't make something up. This is actually the behavior we want from an AI agent instead of inventing an answer, the model recognizes that its knowledge isn't sufficient and needs to use a tool.

Now let's prove the model is also **stateless**.

In [8]:
r1 = client.models.generate_content(model=MODEL, contents="My name is Fatima. Remember that.")
print("Response 1:", r1.text)

r2 = client.models.generate_content(model=MODEL, contents="What's my name?")
print("Response 2:", r2.text)


Response 1: Got it, Fatima! Nice to meet you. I've noted your name and will remember it. How can I help you today?
Response 2: I don't know your name yet! Since I don't have access to your personal information, you'll have to tell me. What should I call you?





The model had no idea, because Call 2 didn't include Call 1's messages. Build a `contents` list
below with **both** turns, then ask "What's my name?" as a 3rd turn. Gemini expects alternating
`user` / `model` turns, each shaped like `{"role": ..., "parts": [{"text": ...}]}`.


In [9]:
# Build conversation history: user → model → user
contents = [
    {
        "role": "user",
        "parts": [
            {"text": "My name is Fatima. Remember that."}
        ]
    },
    {
        "role": "model",
        "parts": [
            {"text": r1.text}
        ]
    },
    {
        "role": "user",
        "parts": [
            {"text": "What's my name?"}
        ]
    }
]

# Send the complete conversation to Gemini
r3 = client.models.generate_content(
    model=MODEL,
    contents=contents
)

print(r3.text)

Your name is Fatima!


**Key takeaway:** every "memory" an LLM app has is really just *you* re-sending history on
every call. This is the backbone of everything below.


---
##  Section 2 — Function Calling Basics: One Worked Example (30 minutes)

We'll build **one** tool together, fully worked, so you see the whole pattern end to end.
Everything after this section, you build yourself from a spec.

**The pattern has 4 parts:**
1. A real Python function that does the work
2. A schema (`FunctionDeclaration`) describing it to the model
3. Send a message with the tool attached — the model asks to call it, it doesn't call it itself
4. Your code executes the function and sends the result back as an "Observation"


In [10]:
# GIVEN — Part 1: the real function
def get_weather(city: str, date: str = "today") -> dict:
    """Pretend weather API — hardcoded data so we don't need a real key."""
    fake_data = {
        "lahore":  {"forecast": "sunny", "temp_c": 34},
        "boston":  {"forecast": "rainy", "temp_c": 18},
        "karachi": {"forecast": "humid", "temp_c": 31},
        "london":  {"forecast": "cloudy", "temp_c": 16},
    }
    data = fake_data.get(city.lower(), {"forecast": "unknown", "temp_c": None})
    return {"city": city, "date": date, **data}

get_weather("Lahore")


{'city': 'Lahore', 'date': 'today', 'forecast': 'sunny', 'temp_c': 34}

In [11]:
# GIVEN — Part 2: the schema
get_weather_declaration = types.FunctionDeclaration(
    name="get_weather",
    description="Get the current or forecasted weather for a specific city.",
    parameters={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name, e.g. Lahore"},
            "date": {"type": "string", "description": "YYYY-MM-DD, or 'today'/'tomorrow'"}
        },
        "required": ["city"]
    }
)

weather_tool = types.Tool(function_declarations=[get_weather_declaration])
config = types.GenerateContentConfig(tools=[weather_tool])


In [12]:
# GIVEN — Part 3: send a message, the model asks to call the tool instead of answering
chat = client.chats.create(model=MODEL, config=config)
response = chat.send_message("What should I wear in Lahore today?")

part = response.candidates[0].content.parts[0]
print("Did the model ask for a tool?", part.function_call is not None)
print(part.function_call)


Did the model ask for a tool? True
id='call_887408' args={'date': 'today', 'city': 'Lahore'} name='get_weather' partial_args=None will_continue=None


### Part 4: execute the tool and send the result back

`part.function_call` has a `.name` and `.args`. You need to:
1. Pull out the function name and arguments
2. Call the **real** `get_weather` function with those arguments
3. Wrap the result in `types.Part.from_function_response(name=..., response={"result": ...})`
4. Send that part back with `chat.send_message(...)`
5. Print the final text answer

Write it below — no scaffolding this time except the comments telling you the steps.


In [14]:
# Part 3 — Ask Gemini a question with the weather tool attached

chat = client.chats.create(
    model=MODEL,
    config=config
)

response = chat.send_message(
    "What is the weather in Lahore today?"
)

# Gemini should return a function call
part = response.candidates[0].content.parts[0]

print("Function requested:", part.function_call.name)
print("Arguments:", part.function_call.args)

# Part 4 — Execute the tool and send the result back

function_call = part.function_call

# 1. Get the function name and arguments
function_name = function_call.name
function_args = dict(function_call.args)

print("Calling:", function_name)
print("With arguments:", function_args)

# 2. Call the real Python function
if function_name == "get_weather":
    tool_result = get_weather(**function_args)
else:
    raise ValueError(f"Unknown function: {function_name}")

print("Tool result:", tool_result)

# 3. Wrap the result as a function response
function_response_part = types.Part.from_function_response(
    name=function_name,
    response={
        "result": tool_result
    }
)

# 4. Send the tool result back to Gemini
final_response = chat.send_message(
    function_response_part
)

# 5. Print Gemini's final answer
print("Final answer:", final_response.text)

Function requested: get_weather
Arguments: {'date': 'today', 'city': 'Lahore'}
Calling: get_weather
With arguments: {'date': 'today', 'city': 'Lahore'}
Tool result: {'city': 'Lahore', 'date': 'today', 'forecast': 'sunny', 'temp_c': 34}
Final answer: The weather in Lahore today is **sunny** with a temperature of **34°C**.


**Test it:** if you did it right, the final answer should mention Lahore's actual sunny/34°C
weather, not a generic answer. If it errors, check the Appendix for the worked solution before
moving on — don't spend more than 5 extra minutes stuck here.


---
##  Section 3 — Build the ReAct Loop Yourself

Section 2 only handles **one** tool call. A real agent loops: call a tool, look at the result,
decide if it needs another tool, and only stop when it has a final answer.

**You are writing this loop from scratch.** Here's the spec:

```
function run_agent(user_query, config, tool_registry, max_steps=5, verbose=True):
    start a chat session with the given config
    send the user_query as the first message

    repeat up to max_steps times:
        look at the parts of the latest response
        collect every part that has a function_call  (there can be more than one!)

        if there are no function_calls:
            print "[Step N] Thought: I have enough info."   (if verbose)
            return the response's text — this is the final answer

        otherwise, for EACH function_call:
            print "[Step N] Action: name(args)"              (if verbose)
            look up the function in tool_registry and call it with the args
            print "[Step N] Observation: result"             (if verbose)
            wrap the result as a function_response Part

        send ALL the function_response Parts back in one chat.send_message() call
        (this becomes the new "response" for the next loop iteration)

    if we run out of steps, return a message saying so
```

Note: Gemini can ask for **multiple tools in one turn** — that's why you collect a *list* of
function_calls per step, not just one.


In [16]:
# implement run_agent() from the spec above.
# tool_registry is a dict like {"get_weather": get_weather}

def run_agent(user_query, config, tool_registry, max_steps=5, verbose=True):
    chat = client.chats.create(model=MODEL, config=config)
    response = chat.send_message(user_query)

    for step in range(max_steps):

        # Extract ALL function calls from the latest response
        function_calls = []

        for part in response.candidates[0].content.parts:
            if part.function_call:
                function_calls.append(part.function_call)

        # If there are no function calls, Gemini has the final answer
        if not function_calls:
            if verbose:
                print(f"[Step {step + 1}] Thought: I have enough info.")
            return response.text

        # Store all tool responses
        function_response_parts = []

        # Execute each requested function
        for function_call in function_calls:

            function_name = function_call.name
            function_args = dict(function_call.args)

            if verbose:
                print(
                    f"[Step {step + 1}] Action: "
                    f"{function_name}({function_args})"
                )

            # Find the function in the registry
            if function_name not in tool_registry:
                raise ValueError(
                    f"Unknown tool requested: {function_name}"
                )

            # Execute the actual Python function
            result = tool_registry[function_name](**function_args)

            if verbose:
                print(
                    f"[Step {step + 1}] Observation: {result}"
                )

            # Wrap the result as a function response
            function_response_parts.append(
                types.Part.from_function_response(
                    name=function_name,
                    response={"result": result}
                )
            )

        # Send ALL tool results back to Gemini
        response = chat.send_message(function_response_parts)

    return "Reached max steps without a final answer."


tool_registry = {"get_weather": get_weather}

**Test your loop:**


In [17]:
answer = run_agent(
    "What is the weather in Lahore today?",
    config,
    tool_registry
)

print("\nFinal Answer:")
print(answer)

[Step 1] Action: get_weather({'date': 'today', 'city': 'Lahore'})
[Step 1] Observation: {'city': 'Lahore', 'date': 'today', 'forecast': 'sunny', 'temp_c': 34}
[Step 2] Thought: I have enough info.

Final Answer:
The weather in Lahore today is **sunny** with a temperature of **34°C**.


**🔧 Checkpoint (3 min):** test that your loop correctly handles a query needing **no** tool
(should print zero `[Step N] Action:` lines and go straight to a final answer).


In [18]:
answer = run_agent("What's the capital of Pakistan?", config=config, tool_registry=tool_registry)
print(answer)


[Step 1] Thought: I have enough info.
The capital of Pakistan is Islamabad.


---
##  Section 4 — Write Two New Tools From a Spec

Your loop from Section 3 works with *any* tools you register with it — that's the whole point of
building it generically. Now prove it: write two brand-new tools, start to finish, no scaffolding.

### Tool A: `calculate`

**Spec:**
- Signature: `calculate(expression: str) -> dict`
- Takes a basic arithmetic expression as a string, e.g. `"12 * 3 + 7"`
- Only allow digits, `+ - * / ( ) .` and spaces — reject anything else (hint: `re.fullmatch` with
  a character-class pattern) and return `{"error": "..."}` if rejected
- Otherwise, evaluate it (hint: Python's `eval()` works fine for this trusted lab context) and
  return `{"expression": expression, "result": <the number>}`
- Wrap the eval in `try/except` and return `{"error": str(e)}` if it fails


In [19]:
import re

def calculate(expression: str) -> dict:
    # Allow only digits, arithmetic operators, parentheses,
    # decimal points, and spaces
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        return {"error": "Invalid characters in expression."}

    try:
        result = eval(expression)
        return {
            "expression": expression,
            "result": result
        }

    except Exception as e:
        return {"error": str(e)}


# Test it
print(calculate("12 * 3 + 7"))
print(calculate("import os"))

{'expression': '12 * 3 + 7', 'result': 43}
{'error': 'Invalid characters in expression.'}


### Tool B: `suggest_activity`

**Spec:**
- Signature: `suggest_activity(weather_condition: str) -> dict`
- Takes a weather keyword (`"sunny"`, `"rainy"`, `"humid"`, `"cloudy"`) and returns a dict with
  keys `"weather_condition"` and `"suggestion"` (a one-line activity idea you write yourself)
- If the keyword isn't one you handled, return a sensible fallback suggestion instead of erroring


In [20]:
# suggest_activity(weather_condition) -> dict, per the spec above

def suggest_activity(weather_condition: str) -> dict:
    condition = weather_condition.lower()

    activities = {
        "sunny": "Go for a walk or have a picnic outdoors.",
        "rainy": "Stay indoors and watch a movie or read a book.",
        "humid": "Relax somewhere cool and try an indoor activity.",
        "cloudy": "Take a comfortable walk or explore a nearby place."
    }

    suggestion = activities.get(
        condition,
        "Choose a relaxing activity that suits the current weather."
    )

    return {
        "weather_condition": weather_condition,
        "suggestion": suggestion
    }


print(suggest_activity("rainy"))
print(suggest_activity("blizzard"))

{'weather_condition': 'rainy', 'suggestion': 'Stay indoors and watch a movie or read a book.'}
{'weather_condition': 'blizzard', 'suggestion': 'Choose a relaxing activity that suits the current weather.'}


### Now write the schemas for both

**Spec for each `FunctionDeclaration`:** `name` matching the function, a specific one-sentence
`description` (this is what the model uses to choose the right tool — vague descriptions cause
wrong tool choices), and a `parameters` object schema matching the function's arguments.


In [21]:
# FunctionDeclaration for calculate

calculate_declaration = types.FunctionDeclaration(
    name="calculate",
    description="Evaluate a basic arithmetic expression and return the numerical result.",
    parameters={
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "A basic arithmetic expression containing numbers and arithmetic operators."
            }
        },
        "required": ["expression"]
    }
)

In [22]:
# FunctionDeclaration for suggest_activity

suggest_activity_declaration = types.FunctionDeclaration(
    name="suggest_activity",
    description="Suggest a suitable activity based on the current weather condition.",
    parameters={
        "type": "object",
        "properties": {
            "weather_condition": {
                "type": "string",
                "description": "The weather condition, such as sunny, rainy, humid, or cloudy."
            }
        },
        "required": ["weather_condition"]
    }
)

### Wire everything together and test chaining

Register both new tools (plus `get_weather` from before) into one `Tool`/config, and one
`tool_registry` dict — same pattern as Section 2, now with 3 entries instead of 1.


In [23]:
# Build one Tool containing all 3 FunctionDeclarations
multi_tool = types.Tool(
    function_declarations=[
        get_weather_declaration,
        calculate_declaration,
        suggest_activity_declaration
    ]
)

# Create Gemini configuration with the multi-tool
config = types.GenerateContentConfig(
    tools=[multi_tool]
)

# Register all 3 real Python functions
tool_registry = {
    "get_weather": get_weather,
    "calculate": calculate,
    "suggest_activity": suggest_activity
}

In [24]:
answer = run_agent(
    "What's the weather in Lahore right now, and suggest something fun to do given that weather?",
    config=config,
    tool_registry=tool_registry
)

print()
print("=" * 50)
print("FINAL ANSWER:", answer)

[Step 1] Action: get_weather({'city': 'Lahore', 'date': 'today'})
[Step 1] Observation: {'city': 'Lahore', 'date': 'today', 'forecast': 'sunny', 'temp_c': 34}
[Step 2] Action: suggest_activity({'weather_condition': 'sunny'})
[Step 2] Observation: {'weather_condition': 'sunny', 'suggestion': 'Go for a walk or have a picnic outdoors.'}
[Step 3] Thought: I have enough info.

FINAL ANSWER: The current weather in Lahore is **sunny** with a temperature of **34°C**.

Since it's sunny outside, a great activity would be to **go for a walk or have a picnic outdoors**! If it feels a bit warm, consider visiting a shaded park or enjoying a morning/evening outing.


**🔧 Checkpoint (3 min):** write a query that should trigger the `calculate` tool instead
(e.g. a hotel-cost-per-night question) and confirm the trace shows `calculate` being called.


In [25]:
# 🔧 write a query that should trigger calculate, then run it
answer = run_agent(
    "A hotel room costs $120 per night. I am staying for 5 nights. How much will the hotel cost in total?",
    config=config,
    tool_registry=tool_registry
)

print(answer)

[Step 1] Action: calculate({'expression': '120 * 5'})
[Step 1] Observation: {'expression': '120 * 5', 'result': 600}
[Step 2] Thought: I have enough info.
The total cost for staying 5 nights at $120 per night will be **$600**.


---
## 🏆 Section 5 — Independent Challenge: Currency Converter (20 minutes)

No spec walkthrough this time — just the requirement. Build a 4th tool, `convert_currency`,
completely on your own, following the same pattern you've now done twice.

**Requirements:**
- Function: `convert_currency(amount: float, from_currency: str, to_currency: str) -> dict`
- Use this fake rate table (hardcode it, real-time rates are out of scope today):
  `{("PKR","USD"): 0.0036, ("USD","PKR"): 278.0, ("PKR","AED"): 0.013, ("AED","PKR"): 76.0}`
- Handle currency pairs not in the table gracefully (return an error dict, don't crash)
- Round the converted amount to 2 decimal places
- Write its `FunctionDeclaration`, register it alongside your other 3 tools, and test it with:
  *"How much is 8500 PKR in USD?"*


In [ ]:
# ✍️ WRITE THIS: the whole convert_currency tool, end to end —
# function, FunctionDeclaration, registration, and a test call.
# (Everything you need is exactly what you did in Sections 2 and 4, just on your own this time.)


In [26]:
# SECTION 5 — Currency Converter Tool

# 1. Define the real Python function
def convert_currency(
    amount: float,
    from_currency: str,
    to_currency: str
) -> dict:

    # Fake exchange-rate table
    rates = {
        ("PKR", "USD"): 0.0036,
        ("USD", "PKR"): 278.0,
        ("PKR", "AED"): 0.013,
        ("AED", "PKR"): 76.0
    }

    # Normalize currency codes
    from_currency = from_currency.upper()
    to_currency = to_currency.upper()

    # Check whether the currency pair exists
    pair = (from_currency, to_currency)

    if pair not in rates:
        return {
            "error": f"Currency pair {from_currency} -> {to_currency} is not supported."
        }

    # Get exchange rate
    rate = rates[pair]

    # Convert and round to 2 decimal places
    converted_amount = round(amount * rate, 2)

    return {
        "amount": amount,
        "from_currency": from_currency,
        "to_currency": to_currency,
        "exchange_rate": rate,
        "converted_amount": converted_amount
    }


# 2. Test the Python function directly
print(convert_currency(8500, "PKR", "USD"))

print(convert_currency(100, "USD", "PKR"))

print(convert_currency(100, "EUR", "USD"))

{'amount': 8500, 'from_currency': 'PKR', 'to_currency': 'USD', 'exchange_rate': 0.0036, 'converted_amount': 30.6}
{'amount': 100, 'from_currency': 'USD', 'to_currency': 'PKR', 'exchange_rate': 278.0, 'converted_amount': 27800.0}
{'error': 'Currency pair EUR -> USD is not supported.'}


In [27]:
convert_currency_declaration = types.FunctionDeclaration(
    name="convert_currency",
    description="Convert an amount from one supported currency to another using the available exchange-rate table.",
    parameters={
        "type": "object",
        "properties": {
            "amount": {
                "type": "number",
                "description": "The amount of money to convert."
            },
            "from_currency": {
                "type": "string",
                "description": "The currency to convert from, such as PKR, USD, or AED."
            },
            "to_currency": {
                "type": "string",
                "description": "The currency to convert to, such as PKR, USD, or AED."
            }
        },
        "required": [
            "amount",
            "from_currency",
            "to_currency"
        ]
    }
)

In [28]:
multi_tool = types.Tool(
    function_declarations=[
        get_weather_declaration,
        calculate_declaration,
        suggest_activity_declaration,
        convert_currency_declaration
    ]
)

config = types.GenerateContentConfig(
    tools=[multi_tool]
)

tool_registry = {
    "get_weather": get_weather,
    "calculate": calculate,
    "suggest_activity": suggest_activity,
    "convert_currency": convert_currency
}

In [29]:
answer = run_agent(
    "How much is 8500 PKR in USD?",
    config=config,
    tool_registry=tool_registry
)

print()
print("=" * 50)
print("FINAL ANSWER:", answer)

[Step 1] Action: convert_currency({'to_currency': 'USD', 'from_currency': 'PKR', 'amount': 8500})
[Step 1] Observation: {'amount': 8500, 'from_currency': 'PKR', 'to_currency': 'USD', 'exchange_rate': 0.0036, 'converted_amount': 30.6}
[Step 2] Thought: I have enough info.

FINAL ANSWER: 8,500 PKR is approximately **30.60 USD** (at an exchange rate of 1 PKR = 0.0036 USD).
